# 🗄️ Course 9 — Exploratory Data Analysis in SQL

> **Platform:** DataCamp | **Track:** Associate Data Analyst in SQL  
> **Tool:** PostgreSQL | **Databases:** stackoverflow, fortune500, evanston311

---

## 📋 About This Course

This course teaches how to explore and understand new datasets using SQL. Using real-world data (Stack Overflow questions, Fortune 500 companies, and city service requests), it covers database exploration, numeric summarization, categorical data cleaning, and date/time analysis.

---

## 📚 Table of Contents

| Chapter | Topic | Databases |
|---------|-------|-----------|
| Chapter 1 | What is in the Database? | stackoverflow, fortune500 |
| Chapter 2 | Summarizing & Aggregating Numeric Data | fortune500, stackoverflow |
| Chapter 3 | Exploring Categorical Data & Unstructured Text | evanston311 |
| Chapter 4 | Working with Dates and Timestamps | evanston311 |

---

## 📌 Chapter 1 — What is in the Database?

---

### Explore Table Sizes
Find which table has the most rows among stackoverflow, company, tag_company, tag_type, fortune500.

In [ ]:
SELECT * FROM stackoverflow LIMIT 5;
-- R: stackoverflow has the most rows

### Count Missing Values
Find how many NULL values exist in key columns of fortune500.

In [ ]:
-- Missing ticker values
SELECT count(*) - COUNT(ticker) AS missing
  FROM fortune500;
-- R: 32

-- Missing industry values
SELECT count(*) - count(industry) AS missing FROM fortune500;
-- R: 13

### Join Tables
Find the shared column between company and fortune500 to join them.

In [ ]:
SELECT company.name
FROM company
INNER JOIN fortune500 ON company.ticker = fortune500.ticker;

### Foreign Keys
> **Q:** Why can't `tag_type.tag` be a foreign key referencing `stackoverflow.tag`?

> **A:** `stackoverflow.tag` contains **duplicate values** — foreign key references require unique values.

### Read an Entity Relationship Diagram
Find the most common stackoverflow tag_type and which companies have that type.

In [ ]:
-- Step 1: Count tags by type
SELECT type, count(*) AS count
FROM tag_type
GROUP BY type
ORDER BY count DESC;

In [ ]:
-- Step 2: Find companies with the 'cloud' tag type
SELECT name, tag_type.tag, tag_type.type
  FROM company
       INNER JOIN tag_company ON company.id = tag_company.company_id
       INNER JOIN tag_type ON tag_company.tag = tag_type.tag
WHERE type = 'cloud';

### COALESCE — Handle Missing Industry Values
Use COALESCE to fall back to sector when industry is NULL, then find the most common value.

In [ ]:
SELECT coalesce(industry, sector, 'Unknown') AS industry2,
       count(*) AS count_industry
FROM fortune500 
GROUP BY industry2
ORDER BY count_industry DESC
LIMIT 1;

### Effects of CAST()
See how casting changes values and explore integer vs numeric division.

In [ ]:
-- Cast profits_change to integer (truncates decimals)
SELECT profits_change, 
       CAST(profits_change AS INTEGER) AS profits_change_int
  FROM fortune500;
-- Example: -7.2 → -7

-- Integer division vs numeric division
SELECT 10/3,          -- Result: 3 (integer)
       10::numeric/3; -- Result: 3.333... (numeric)

-- Cast text to numeric
SELECT '3.2'::numeric, '-123'::numeric, '1e3'::numeric,
       '1e-3'::numeric, '02314'::numeric, '0002'::numeric;

### Summarize Distribution of Numeric Values
Explore the distribution of revenues_change and count companies with positive growth.

In [ ]:
-- Raw distribution
SELECT revenues_change, count(*)
FROM fortune500
GROUP BY revenues_change
ORDER BY revenues_change ASC;

-- Cast as integer for simpler view
SELECT revenues_change::integer, count(*)
FROM fortune500
GROUP BY revenues_change::integer
ORDER BY revenues_change ASC;

-- How many companies had revenue increase?
SELECT COUNT(*)
FROM fortune500
WHERE revenues_change > 0;
-- R: 298

---

## 📌 Chapter 2 — Summarizing & Aggregating Numeric Data

---

### Division — Revenue per Employee
Compute average revenue per employee by sector.

In [ ]:
SELECT sector, 
       AVG(revenues/employees::numeric) AS avg_rev_employee
  FROM fortune500
 GROUP BY sector
 ORDER BY AVG(revenues/employees::numeric);

### Explore with Division
Verify whether unanswered_pct = unanswered_count / question_count.

In [ ]:
SELECT unanswered_count/question_count::numeric AS computed_pct, 
       unanswered_pct
FROM stackoverflow
WHERE question_count <> 0
LIMIT 10;

### Summarize Numeric Columns — MIN, AVG, MAX, STDDEV
Summarize profits in fortune500, overall and by sector.

In [ ]:
-- Overall summary
SELECT MIN(profits), AVG(profits), MAX(profits), stddev(profits)
  FROM fortune500;

-- By sector
SELECT sector, MIN(profits), AVG(profits), MAX(profits), stddev(profits)
FROM fortune500
GROUP BY sector
ORDER BY avg;

### Summarize Group Statistics
Compute the stddev, min, max, and avg of the maximum question_count per tag.

In [ ]:
SELECT stddev(maxval), MIN(maxval), MAX(maxval), AVG(maxval)
  FROM (
        SELECT MAX(question_count) AS maxval
        FROM stackoverflow
        GROUP BY tag) AS max_results;

### TRUNC() — Creating Bins
Use trunc() to group Fortune 500 companies into employee size bins.

In [ ]:
-- Bin by 100,000s
SELECT trunc(employees, -5) AS employee_bin, count(employees)
FROM fortune500
GROUP BY employee_bin
ORDER BY employee_bin;

-- Bin by 10,000s (companies < 100,000 employees)
SELECT trunc(employees, -4) AS employee_bin, count(employees)
FROM fortune500
WHERE employees < 100000
GROUP BY employee_bin
ORDER BY employee_bin;

### GENERATE_SERIES() — Custom Bins
Bin the distribution of Stack Overflow 'dropbox' question counts into groups of 50.

In [ ]:
-- Find range
SELECT MIN(question_count), MAX(question_count)
  FROM stackoverflow
 WHERE tag = 'dropbox';
-- R: min=2315, max=3072

In [ ]:
-- Create bins and count values in each
WITH bins AS (
      SELECT generate_series(2200, 3050, 50) AS lower,
             generate_series(2250, 3100, 50) AS upper),
     dropbox AS (
      SELECT question_count 
        FROM stackoverflow
       WHERE tag = 'dropbox') 
SELECT lower, upper, count(question_count) 
  FROM bins
       LEFT JOIN dropbox ON question_count >= lower AND question_count < upper
 GROUP BY lower, upper
 ORDER BY lower;

### CORR() — Correlation Between Financial Metrics
Compute correlations between Fortune 500 revenues and other financial variables.

In [ ]:
SELECT corr(revenues, profits)  AS rev_profits,
       corr(revenues, assets)   AS rev_assets,
       corr(revenues, equity)   AS rev_equity 
  FROM fortune500;

### PERCENTILE_DISC() — Mean and Median
Compute mean and median assets per sector.

In [ ]:
SELECT sector,
       AVG(assets) AS mean,
       percentile_disc(0.5) WITHIN GROUP (ORDER BY assets) AS median
  FROM fortune500
 GROUP BY sector
 ORDER BY AVG(assets);

### CREATE TEMP TABLE — Top 20% Profits by Sector
Find Fortune 500 companies in the top 20% of profits for their sector.

In [ ]:
DROP TABLE IF EXISTS profit80;

CREATE TEMP TABLE profit80 AS
  SELECT sector, 
         percentile_disc(0.8) WITHIN GROUP (ORDER BY profits) AS pct80
    FROM fortune500 
   GROUP BY sector;

-- Select companies above the 80th percentile
SELECT title, fortune500.sector, profits,
       profits/pct80 AS ratio
  FROM fortune500 
  LEFT JOIN profit80 ON fortune500.sector = profit80.sector
 WHERE profits > pct80;

### TEMP TABLE — Tag Growth Over Time
Compare question counts on the first and last date for each Stack Overflow tag.

In [ ]:
DROP TABLE IF EXISTS startdates;

CREATE TEMP TABLE startdates AS
SELECT tag, min(date) AS mindate
  FROM stackoverflow
 GROUP BY tag;

SELECT startdates.tag, mindate, 
       so_min.question_count AS min_date_question_count,
       so_max.question_count AS max_date_question_count,
       so_max.question_count - so_min.question_count AS change
  FROM startdates
       INNER JOIN stackoverflow AS so_min
          ON startdates.tag = so_min.tag AND startdates.mindate = so_min.date
       INNER JOIN stackoverflow AS so_max
          ON startdates.tag = so_max.tag AND so_max.date = '2018-09-25';

### INSERT INTO TEMP TABLE — Correlation Matrix
Build a full correlation matrix between profits, profits_change, and revenues_change.

In [ ]:
DROP TABLE IF EXISTS correlations;

CREATE TEMP TABLE correlations AS
SELECT 'profits'::varchar AS measure,
       corr(profits, profits) AS profits,
       corr(profits, profits_change) AS profits_change,
       corr(profits, revenues_change) AS revenues_change
  FROM fortune500;

INSERT INTO correlations
SELECT 'profits_change'::varchar AS measure,
       corr(profits_change, profits) AS profits,
       corr(profits_change, profits_change) AS profits_change,
       corr(profits_change, revenues_change) AS revenues_change
  FROM fortune500;

INSERT INTO correlations
SELECT 'revenues_change'::varchar AS measure,
       corr(revenues_change, profits) AS profits,
       corr(revenues_change, profits_change) AS profits_change,
       corr(revenues_change, revenues_change) AS revenues_change
  FROM fortune500;

-- View the correlation matrix
SELECT measure, 
       round(profits::numeric, 2)          AS profits,
       round(profits_change::numeric, 2)   AS profits_change,
       round(revenues_change::numeric, 2)  AS revenues_change
FROM correlations;

---

## 📌 Chapter 3 — Exploring Categorical Data & Unstructured Text

---

### Count the Categories
Explore frequency of values in key columns of the evanston311 table.

In [ ]:
-- Count by priority level
SELECT priority, COUNT(*) FROM evanston311 GROUP BY priority;

-- Zip codes in at least 100 rows
SELECT DISTINCT zip, count(*)
FROM evanston311
GROUP BY zip HAVING count(*) >= 100;

-- Sources in at least 100 rows
SELECT DISTINCT source, count(*)
FROM evanston311
GROUP BY source HAVING COUNT(*) >= 100;

-- Top 5 most common streets
SELECT street, count(*)
FROM evanston311
GROUP BY street ORDER BY COUNT DESC LIMIT 5;

### TRIM() — Cleaning Street Values
Remove house numbers, extra punctuation, and spaces from street values.

In [ ]:
SELECT DISTINCT street,
       trim(street, '0123456789 #/.') AS cleaned_street
FROM evanston311
ORDER BY street;

### ILIKE / LIKE — Exploring Unstructured Text
Find descriptions mentioning trash/garbage that aren't in trash-related categories.

In [ ]:
-- Count descriptions with trash/garbage
SELECT count(*) FROM evanston311
WHERE description ILIKE '%trash%' OR description ILIKE '%garbage%';
-- R: 2551

-- Count mismatches (description mentions trash but category doesn't)
SELECT count(*)
FROM evanston311 
WHERE (description ILIKE '%trash%' OR description ILIKE '%garbage%')
  AND category NOT LIKE '%Trash%'
  AND category NOT LIKE '%Garbage%';
-- R: 570

-- Most common categories for those mismatches
SELECT category, count(*)
  FROM evanston311 
WHERE (description ILIKE '%trash%' OR description ILIKE '%garbage%') 
  AND category NOT LIKE '%Trash%'
  AND category NOT LIKE '%Garbage%'
GROUP BY category ORDER BY count DESC LIMIT 10;

### LTRIM + CONCAT — Build Full Address
Concatenate house_num and street into a clean address.

In [ ]:
SELECT LTRIM(CONCAT(house_num, ' ', street)) AS address
  FROM evanston311;

### SPLIT_PART() — Extract Street Name
Extract just the first word of each street to find the most common streets regardless of suffix.

In [ ]:
SELECT split_part(street, ' ', 1) AS street_name, count(*)
FROM evanston311
GROUP BY street_name
ORDER BY count DESC
LIMIT 20;

### LEFT() + CASE — Shorten Long Strings
Display the first 50 characters of descriptions starting with 'I', adding '...' when truncated.

In [ ]:
SELECT CASE WHEN length(description) > 50
            THEN left(description, 50) || '...'
       ELSE description END
  FROM evanston311
 WHERE description LIKE 'I %'
 ORDER BY description;

### CASE WHEN — Create 'Other' Category
> **Q:** Which threshold groups low-frequency zip codes into 'other' and produces the result shown?

> **A: 100** — zip codes with fewer than 100 requests are grouped as 'other'.

### TEMP TABLE + UPDATE — Recode Categories
Standardize ~150 category values by extracting the main category before the dash.

In [ ]:
DROP TABLE IF EXISTS recode;

CREATE TEMP TABLE recode AS
  SELECT DISTINCT category, 
         RTRIM(split_part(category, '-', 1)) AS standardized
    FROM evanston311;

-- Clean up similar values
UPDATE recode SET standardized = 'Trash Cart'   WHERE standardized LIKE 'Trash%Cart';
UPDATE recode SET standardized = 'Snow Removal' WHERE standardized LIKE 'Snow%Removal%';
UPDATE recode SET standardized = 'UNUSED'
 WHERE standardized IN ('THIS REQUEST IS INACTIVE...Trash Cart', 
                        '(DO NOT USE) Water Bill', 
                        'DO NOT USE Trash', 'NO LONGER IN USE');

-- Join to count by standardized category
SELECT standardized, count(*)
  FROM evanston311 
       LEFT JOIN recode ON evanston311.category = recode.category
 GROUP BY standardized
 ORDER BY count DESC;

### CAST Boolean as Integer — Indicator Variables
Create email and phone indicator columns and analyze contact info by priority.

In [ ]:
DROP TABLE IF EXISTS indicators;

CREATE TEMP TABLE indicators AS
  SELECT id, 
         CAST(description LIKE '%@%' AS integer) AS email,
         CAST(description LIKE '%___-___-____%' AS integer) AS phone 
    FROM evanston311;

-- Proportion of email/phone mentions by priority
SELECT priority,
       sum(email)/count(*)::numeric AS email_prop, 
       sum(phone)/count(*)::numeric AS phone_prop
  FROM evanston311
       LEFT JOIN indicators ON evanston311.id = indicators.id
 GROUP BY priority;

---

## 📌 Chapter 4 — Working with Dates and Timestamps

---

### Date Comparisons
Dates auto-convert to timestamps at midnight — use CAST or range operators to match correctly.

In [ ]:
-- Cast to date for exact day match
SELECT count(*) FROM evanston311
 WHERE CAST(date_created AS date) = '2017-01-31';

-- Use >= and < for a date range
SELECT count(*) FROM evanston311
 WHERE date_created >= '2016-02-29' AND date_created < '2016-03-01';

-- Add 1 day to upper bound
SELECT count(*) FROM evanston311
 WHERE date_created >= '2017-03-13'
   AND date_created < '2017-03-13'::date + 1;

### Date Arithmetic
Subtract dates, use now(), and add intervals to timestamps.

In [ ]:
-- Range of data
SELECT MAX(date_created) - MIN(date_created) FROM evanston311;

-- Age of most recent request
SELECT now() - max(date_created) FROM evanston311;

-- 100 days from now
SELECT now() + '100 days'::interval;

-- Now and 5 minutes from now
SELECT now(), now() + '5 minutes'::interval;

### Completion Time by Category
Find which category takes the longest to resolve.

In [ ]:
SELECT category, 
       AVG(date_completed - date_created) AS completion_time
FROM evanston311
GROUP BY category
ORDER BY completion_time DESC;

### DATE_PART() — Extracting Date Components
Count requests by month, find the most common hour, and count completions by hour.

In [ ]:
-- Requests per month in 2016-2017
SELECT date_part('month', date_created) AS month, count(*)
FROM evanston311
WHERE date_created >= '2016-01-01' AND date_created < '2018-01-01'
GROUP BY month;

-- Most common creation hour
SELECT date_part('hour', date_created) AS hour, count(*)
FROM evanston311
GROUP BY hour ORDER BY count DESC LIMIT 1;
-- R: Hour 11 (3960 requests)

-- Completions by hour
SELECT date_part('hours', date_completed) AS hour, count(*)
FROM evanston311
GROUP BY hour ORDER BY hour;

### TO_CHAR() + EXTRACT(DOW) — Day of Week Analysis
Compute average completion time by day of week.

In [ ]:
SELECT to_char(date_created, 'day') AS day, 
       AVG(date_completed - date_created) AS duration
FROM evanston311 
GROUP BY day, EXTRACT(DOW FROM date_created)
ORDER BY EXTRACT(DOW FROM date_created);

### DATE_TRUNC() — Monthly Averages
Compute average daily requests per month using date_trunc().

In [ ]:
SELECT date_trunc('month', day) AS month, AVG(count)
  FROM (SELECT date_trunc('day', date_created) AS day, count(*) AS count
          FROM evanston311
         GROUP BY day) AS daily_count
 GROUP BY month ORDER BY month;

### GENERATE_SERIES() — Find Missing Dates
Identify days with no service requests using generate_series.

In [ ]:
SELECT day
  FROM (SELECT generate_series(min(date_created),
                               max(date_created),
                               '1 day')::date AS day
          FROM evanston311) AS all_dates
 WHERE day NOT IN 
       (SELECT date_created::date FROM evanston311);

### Custom Aggregation Periods
Compute the median daily requests per 6-month period using generate_series bins.

In [ ]:
WITH bins AS (
    SELECT generate_series('2016-01-01', '2018-01-01', '6 months'::interval) AS lower,
           generate_series('2016-07-01', '2018-07-01', '6 months'::interval) AS upper),
     daily_counts AS (
      SELECT day, count(date_created) AS count
      FROM (SELECT generate_series('2016-01-01', '2018-06-30',
                                   '1 day'::interval)::date AS day) AS daily_series
       LEFT JOIN evanston311 ON day = date_created::date
      GROUP BY day)

SELECT lower, upper, 
       percentile_disc(0.5) WITHIN GROUP (ORDER BY count) AS median
    FROM bins
         LEFT JOIN daily_counts ON day >= lower AND day < upper
   GROUP BY lower, upper ORDER BY lower;

### Monthly Average Including Missing Dates
Use COALESCE to replace NULL counts with 0 when no requests were made on a day.

In [ ]:
WITH all_days AS 
     (SELECT generate_series('2016-01-01', '2018-06-30', '1 day'::interval) AS date),
     daily_count AS 
     (SELECT date_trunc('day', date_created) AS day, count(*) AS count
      FROM evanston311 GROUP BY day)

SELECT date_trunc('month', date) AS month,
       avg(coalesce(count, 0)) AS average
FROM all_days
       LEFT JOIN daily_count ON all_days.date = daily_count.day
 GROUP BY month ORDER BY month;

### LAG() — Longest Gap Between Requests
Find the maximum time gap between consecutive service requests.

In [ ]:
WITH request_gaps AS (
        SELECT date_created,
               lag(date_created) OVER (ORDER BY date_created) AS previous,
               date_created - lag(date_created) OVER (ORDER BY date_created) AS gap
          FROM evanston311)
SELECT *
  FROM request_gaps
 WHERE gap = (SELECT MAX(gap) FROM request_gaps);

### Final Investigation — Rats!
Why do 'Rodents-Rats' requests average 64 days to resolve? A 4-step investigation.

In [ ]:
-- Step 1: Distribution of completion times
SELECT date_trunc('day', date_completed - date_created) AS completion_time, count(*)
  FROM evanston311
 WHERE category = 'Rodents- Rats'
 GROUP BY completion_time ORDER BY completion_time;

In [ ]:
-- Step 2: Average time excluding top 5% outliers
SELECT category, AVG(date_completed - date_created) AS avg_completion_time
  FROM evanston311
 WHERE (date_completed - date_created) < 
         (SELECT percentile_disc(0.95) WITHIN GROUP 
                 (ORDER BY (date_completed - date_created)) FROM evanston311)
 GROUP BY category ORDER BY avg_completion_time DESC;

In [ ]:
-- Step 3: Correlation between monthly avg completion time and request count
SELECT corr(avg_completion, count)
  FROM (SELECT date_trunc('month', date_created) AS month, 
               AVG(EXTRACT(epoch FROM date_completed - date_created)) AS avg_completion, 
               count(*) AS count
         FROM evanston311
         WHERE category = 'Rodents- Rats' 
         GROUP BY month) AS monthly_avgs;

In [ ]:
-- Step 4: Monthly created vs completed requests
WITH created AS (
       SELECT date_trunc('month', date_created) AS month, count(*) AS created_count
         FROM evanston311 WHERE category = 'Rodents- Rats' GROUP BY month),
      completed AS (
       SELECT date_trunc('month', date_completed) AS month, count(*) AS completed_count
         FROM evanston311 WHERE category = 'Rodents- Rats' GROUP BY month)

SELECT created.month, created_count, completed_count
FROM created
INNER JOIN completed ON created.month = completed.month
ORDER BY created.month;